In [1]:
import sys
sys.path.append('..')

import time
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

from analysis import (
    load_data,
    get_city_data,
    add_rolling_stats,
    detect_anomalies,
    compute_seasonal_stats
)

In [2]:
df = load_data('../data/temperature_data.csv')
cities = df['city'].unique().tolist()
print(f"Городов: {len(cities)}")
print(f"Всего записей: {len(df)}")
print(f"Города: {cities}")

Городов: 15
Всего записей: 54750
Города: ['Beijing', 'Berlin', 'Cairo', 'Dubai', 'London', 'Los Angeles', 'Mexico City', 'Moscow', 'Mumbai', 'New York', 'Paris', 'Rio de Janeiro', 'Singapore', 'Sydney', 'Tokyo']


In [3]:
def analyze_city(city_df):
    city_df = add_rolling_stats(city_df)
    city_df = detect_anomalies(city_df)
    seasonal_stats = compute_seasonal_stats(city_df)
    return city_df, seasonal_stats


def analyze_city_from_df(args):
    df, city = args
    city_df = get_city_data(df, city)
    return city, *analyze_city(city_df)

In [4]:
def analyze_sequential(df, cities):
    """Последовательный анализ всех городов."""
    results = {}
    for city in cities:
        city_df = get_city_data(df, city)
        city_df, seasonal_stats = analyze_city(city_df)
        results[city] = {'data': city_df, 'seasonal_stats': seasonal_stats}
    return results

start = time.time()
results_seq = analyze_sequential(df, cities)
time_seq = time.time() - start

print(f"Последовательно: {time_seq:.4f} сек")

Последовательно: 0.0665 сек


In [5]:
def analyze_parallel_threads(df, cities, max_workers=4):
    """Параллельный анализ с ThreadPoolExecutor."""
    args = [(df, city) for city in cities]
    results = {}
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for city, city_df, seasonal_stats in executor.map(analyze_city_from_df, args):
            results[city] = {'data': city_df, 'seasonal_stats': seasonal_stats}
    
    return results

# Замер времени
start = time.time()
results_threads = analyze_parallel_threads(df, cities)
time_threads = time.time() - start

print(f"Параллельно (потоки): {time_threads:.4f} сек")

Параллельно (потоки): 0.0620 сек


In [6]:
print("РЕЗУЛЬТАТЫ ЭКСПЕРИМЕНТА")
print("=" * 50)
print(f"Последовательно: {time_seq:.4f} сек")
print(f"Параллельно (потоки): {time_threads:.4f} сек")
print("")
print(f"Ускорение (потоки): {time_seq / time_threads:.2f}x")

РЕЗУЛЬТАТЫ ЭКСПЕРИМЕНТА
Последовательно: 0.0665 сек
Параллельно (потоки): 0.0620 сек

Ускорение (потоки): 1.07x


## Выводы по распараллеливанию

### Результаты:
| Метод | Время | Ускорение |
|-------|-------|-----------|
| Последовательно | 0.0665-0.0752 сек | — |
| Параллельно (ThreadPoolExecutor) | 0.0620-0.0768 сек | 0.98-1.07x |

=> параллелизация почти не дала ускорения. Почему?

1. Малый объём вычислений - анализ 15 городов занял ~60-75 мс. В данном случае  расходы ресурсов на создание потоков и синхронизацию сопоставимы с самой работой.

2. ThreadPoolExecutor более полезен в I/O задачах: сетевых запросах или при чтении файлов
   - Пример: параллельные запросы к API OpenWeatherMap для нескольких городов

**Вывод:** Для данной задачи (анализ ~55 000 записей) последовательное выполнение оптимально. Параллелизация имеет смысл для (API-запросов) или при значительно большем объёме данных.